# 03 — The antenna response

The response of a GRAND antenna to an incoming radio wave is its **effective
length** $\boldsymbol{\ell}$: a complex, direction- and frequency-dependent
vector. The open-circuit voltage induced at one arm is its projection onto the
electric field,

$$V_{\rm oc}^{\,p}(\nu) \;=\; \boldsymbol{\ell}^{\,p}(\nu,\theta,\phi) \cdot \boldsymbol{E}(\nu),$$

with $p \in \{\rm SN, EW, Z\}$ running over the three arms of a HorizonAntenna.

This is the first stage of the simulation chain and the one that carries the
most instrument-specific knowledge: nothing outside GRAND knows what a
HorizonAntenna is. Everything downstream — the RF chain, the ADC — is generic
electronics; **this** is the physics of the detector.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from grand.sim.detector.antenna_model import AntennaModel

model = AntennaModel()
print("arms:", list(model.d_leff))

## 1. What is actually tabulated

Each arm is a `DataTable` holding the response on a regular grid. The two
complex arrays `leff_theta_reim` and `leff_phi_reim` are the $\theta$ and
$\phi$ components of $\boldsymbol{\ell}$ in the local spherical basis — the
antenna is described in its own frame, and it is `process_ant.py` that rotates
the shower direction into that frame before interpolating.

Note that `leff_theta` / `leff_phi` (magnitude) and `phase_theta` / `phase_phi`
exist as attributes but are `None`: the tables ship in real/imaginary form and
the polar form is never populated.

In [ ]:
t = model.leff_sn
print("frequency  ", t.frequency.shape, "  %.0f - %.0f MHz" % (t.frequency.min()/1e6, t.frequency.max()/1e6))
print("phi        ", t.phi.shape, "      %.0f - %.0f deg" % (t.phi.min(), t.phi.max()))
print("theta      ", t.theta.shape, "       %.0f - %.0f deg" % (t.theta.min(), t.theta.max()))
print("leff_theta_reim", t.leff_theta_reim.shape, t.leff_theta_reim.dtype)
print()
print("polar form populated? leff_theta =", t.leff_theta, " phase_theta =", t.phase_theta)

## 2. Effective length against frequency

At a fixed arrival direction, $|\ell_\theta|$ across the 30–250 MHz band. This
curve is what decides which part of the shower spectrum the antenna is
sensitive to; combined with the Galactic background of notebook 05, it sets the
band the experiment actually operates in.

In [ ]:
freq_mhz = t.frequency / 1e6
i_phi   = int(np.argmin(np.abs(t.phi - 45.0)))     # 45 deg azimuth
i_theta = int(np.argmin(np.abs(t.theta - 60.0)))   # 60 deg zenith: a typical inclined shower

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for name, arm in model.d_leff.items():
    ax[0].plot(freq_mhz, np.abs(arm.leff_theta_reim[:, i_phi, i_theta]), label=name)
    ax[1].plot(freq_mhz, np.abs(arm.leff_phi_reim[:, i_phi, i_theta]),   label=name)
ax[0].set_ylabel(r'$|\ell_\theta|$ [m]'); ax[1].set_ylabel(r'$|\ell_\phi|$ [m]')
for a in ax:
    a.set_xlabel('frequency [MHz]'); a.grid(alpha=.3); a.legend()
fig.suptitle(r'effective length at $\phi=45^\circ$, $\theta=60^\circ$')
fig.tight_layout()

Two things to read off this figure. First, the resonance near 100–150 MHz:
that is where the arms are electrically a useful fraction of a wavelength.
Second, **the Z arm is not a scaled copy of the horizontal arms** — it has a
different shape entirely, and it carries almost all of its response in
$\ell_\theta$. That asymmetry propagates all the way to the noise budget.

## 3. The directional map

Fixing the frequency instead and sweeping direction gives the beam pattern.
Figure 5 of [arXiv:2408.10926](https://arxiv.org/abs/2408.10926) shows the same
quantity.

In [ ]:
i_f = int(np.argmin(np.abs(freq_mhz - 150.0)))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), subplot_kw=dict(projection='polar'))
PHI, TH = np.meshgrid(np.radians(t.phi), t.theta, indexing='ij')
for ax, (name, arm) in zip(axes, model.d_leff.items()):
    amp = np.abs(arm.leff_theta_reim[i_f])          # (phi, theta)
    pc = ax.pcolormesh(PHI, TH, amp, cmap='viridis', shading='auto')
    ax.set_title(r'%s,  $|\ell_\theta|$ at %.0f MHz' % (name, freq_mhz[i_f]), pad=14)
    ax.set_rlabel_position(135)
    fig.colorbar(pc, ax=ax, pad=.10, label='[m]')
fig.tight_layout()

The radial coordinate is zenith angle: the centre of each disc is straight up,
the rim is the horizon. The horizontal arms show the expected two-lobed pattern
aligned with the arm; the Z arm is azimuthally symmetric and blind at zenith —
a vertical dipole cannot respond to a wave arriving from directly overhead.

## 4. Comparing the arms quantitatively

In [ ]:
print("%-5s %10s %10s %10s" % ("arm", "peak", "median", "at MHz"))
for name, arm in model.d_leff.items():
    amp = np.abs(arm.leff_theta_reim)
    i_pk = np.unravel_index(amp.argmax(), amp.shape)
    print("%-5s %9.2f m %8.3f m %9.0f" % (name, amp.max(), np.median(amp), freq_mhz[i_pk[0]]))

## 5. Phase matters too

The effective length is complex, so the antenna does not merely scale the field
— it disperses it. Unwrapping the phase along frequency gives a group delay;
ignoring it would smear the sub-nanosecond timing that GRAND's reconstruction
depends on.

In [ ]:
z = t.leff_theta_reim[:, i_phi, i_theta]
phase = np.unwrap(np.angle(z))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(freq_mhz, np.degrees(phase))
ax.set_xlabel('frequency [MHz]'); ax.set_ylabel('phase [deg]')
ax.set_title('unwrapped phase of $\\ell_\\theta$, SN arm'); ax.grid(alpha=.3)
fig.tight_layout()

band = (freq_mhz > 50) & (freq_mhz < 200)
slope = np.polyfit(t.frequency[band], phase[band], 1)[0]
print("group delay over 50-200 MHz: %.2f ns" % (-slope / (2 * np.pi) * 1e9))

## 6. A note for contributors

`AntennaModel.plot_effective_length()` exists but its body is `pass` — it is a
stub. The plots above are what it was presumably meant to produce. If you fill
it in, the figures in this notebook are a reasonable specification.

## Where next

- [02 — Reading and writing GRAND data](02_data_model.ipynb)
- [04 — The RF chain](04_rf_chain.ipynb) — what happens to $V_{\rm oc}$ next
- [05 — Galactic noise](05_galactic_noise.ipynb) — why the Z arm matters